# Bluestock Mutual Fund Analytics EDA

 

This notebook performs end-to-end Exploratory Data Analysis for the Mutual Fund Analytics Capstone project. It loads cleaned datasets, generates 15+ visualizations, exports PNG charts, and captures observations and business insights.

 

**Key analysis areas:**

- NAV trends

- AUM growth

- SIP inflow dynamics

- Category-level heatmap

- Investor demographics

- State-level transaction analysis

- Top vs bottom performing funds

- Folio growth

- Fund performance comparison

- Benchmark and risk analysis

- Sector allocation


In [ ]:
import pandas as pd

import numpy as np

import matplotlib.pyplot as plt

import seaborn as sns

from pathlib import Path



sns.set_theme(style='whitegrid', font_scale=1.08)

plt.rcParams['figure.max_open_warning'] = 20



ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path.cwd()

RAW_DIR = ROOT / 'data' / 'raw'

PROCESSED_DIR = ROOT / 'data' / 'processed'

CHARTS_DIR = ROOT / 'reports' / 'charts'

CHARTS_DIR.mkdir(parents=True, exist_ok=True)



DATA_FILES = {

    'fund_master': '01_fund_master.csv',

    'nav_history': '02_nav_history.csv',

    'aum_by_fund_house': '03_aum_by_fund_house.csv',

    'monthly_sip_inflows': '04_monthly_sip_inflows.csv',

    'category_inflows': '05_category_inflows.csv',

    'industry_folio_count': '06_industry_folio_count.csv',

    'scheme_performance': '07_scheme_performance.csv',

    'investor_transactions': '08_investor_transactions.csv',

    'portfolio_holdings': '09_portfolio_holdings.csv',

    'benchmark_indices': '10_benchmark_indices.csv',

}



def load_csv(name: str) -> pd.DataFrame:

    path = PROCESSED_DIR / DATA_FILES[name]

    if not path.exists():

        path = RAW_DIR / DATA_FILES[name]

    if not path.exists():

        raise FileNotFoundError(f'Missing file: {path}')

    df = pd.read_csv(path)

    return normalize_columns(df)



def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:

    df.columns = [str(c).strip().lower().replace(' ', '_') for c in df.columns]

    return df



def save_chart(fig, filename: str) -> None:

    output_path = CHARTS_DIR / filename

    fig.savefig(output_path, dpi=220, bbox_inches='tight')

    print(f'Saved chart: {output_path}')


In [ ]:
datasets = {}
for key in DATA_FILES:
    try:
        datasets[key] = load_csv(key)
        print(f'Loaded {key}: {datasets[key].shape}')
    except FileNotFoundError as exc:
        print(str(exc))

for name, df in datasets.items():
    print('\n=== Dataset summary:', name, '===')
    print('Columns:', list(df.columns))
    print('Missing values:')
    print(df.isnull().sum().sort_values(ascending=False).head(15))
    print('Duplicate rows:', df.duplicated().sum())


In [ ]:
nav = datasets.get('nav_history', pd.DataFrame())
aum = datasets.get('aum_by_fund_house', pd.DataFrame())

if not nav.empty:
    nav['date'] = pd.to_datetime(nav['date'], errors='coerce')
    nav_trend = nav.groupby('date')['nav'].mean().sort_index()
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(nav_trend.index, nav_trend.values, marker='o', linewidth=1.5, color='#0b3d91')
    ax.set_title('Average NAV Trend')
    ax.set_xlabel('Date')
    ax.set_ylabel('Average NAV')
    ax.grid(alpha=0.3)
    save_chart(fig, 'nav_trend_analysis.png')
    plt.show()
    print('Observation: Average NAV is plotted over time based on the available scheme history. \nInsight: Seasonal NAV movement may reveal cyclic fund performance and investor sentiment.')

if not aum.empty:
    aum['date'] = pd.to_datetime(aum['date'], errors='coerce')
    aum_sorted = aum.sort_values('aum_crore', ascending=False).head(12)
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.barplot(data=aum_sorted, x='aum_crore', y='fund_house', palette='viridis', ax=ax)
    ax.set_title('Top Fund Houses by AUM')
    ax.set_xlabel('AUM (Crore)')
    ax.set_ylabel('Fund House')
    save_chart(fig, 'aum_growth_analysis.png')
    plt.show()
    print('Observation: The chart highlights fund houses with the largest assets under management. \nInsight: High AUM concentration indicates strong market leadership and investor trust.')


In [ ]:
sip = datasets.get('monthly_sip_inflows', pd.DataFrame())
category = datasets.get('category_inflows', pd.DataFrame())

if not sip.empty:
    sip['month'] = pd.to_datetime(sip['month'], errors='coerce')
    sip_plot = sip.sort_values('month')
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(sip_plot['month'], sip_plot['sip_inflow_crore'], marker='o', color='#1f77b4')
    ax.set_title('Monthly SIP Inflow Trend')
    ax.set_xlabel('Month')
    ax.set_ylabel('SIP Inflow (Crore)')
    ax.grid(True, alpha=0.3)
    save_chart(fig, 'sip_inflow_trend.png')
    plt.show()
    print('Observation: Monthly SIP inflow trend tracks investor momentum. \nInsight: Increasing SIP inflows suggest strong retail participation and systematic investment adoption.')

if not category.empty:
    pivot = category.pivot(index='category', columns='month', values='net_inflow_crore')
    fig, ax = plt.subplots(figsize=(14, 8))
    sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlGnBu', ax=ax)
    ax.set_title('Category Inflow Heatmap')
    ax.set_xlabel('Month')
    ax.set_ylabel('Category')
    save_chart(fig, 'category_inflow_heatmap.png')
    plt.show()
    print('Observation: The heatmap shows category inflow intensity over time. \nInsight: Category-level shifts can signal where investor preferences are moving across equity, debt, and hybrid segments.')


In [ ]:
investor = datasets.get('investor_transactions', pd.DataFrame())

if not investor.empty:
    investor['transaction_date'] = pd.to_datetime(investor['transaction_date'], errors='coerce')
    investor['age_group'] = investor['age_group'].astype(str).str.strip()
    age_counts = investor['age_group'].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.pie(age_counts, labels=age_counts.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette('pastel'))
    ax.set_title('Investor Age Group Distribution')
    save_chart(fig, 'investor_age_group_pie.png')
    plt.show()
    print('Observation: Age group distribution reveals the investor mix. \nInsight: Identifying dominant age cohorts helps tailor fund outreach and SIP offerings.')

    gender_counts = investor['gender'].fillna('Unknown').value_counts()
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.barplot(x=gender_counts.index, y=gender_counts.values, palette='Set2', ax=ax)
    ax.set_title('Investor Gender Distribution')
    ax.set_xlabel('Gender')
    ax.set_ylabel('Count')
    save_chart(fig, 'investor_gender_distribution.png')
    plt.show()
    print('Observation: Gender split indicates audience balance. \nInsight: Strong gender representation can inform targeted education and marketing programs.')


In [ ]:
if not investor.empty:
    state_counts = investor['state'].fillna('Unknown').value_counts().nlargest(15)
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.barplot(x=state_counts.values, y=state_counts.index, palette='magma', ax=ax)
    ax.set_title('Top States by Investor Transactions')
    ax.set_xlabel('Transaction Count')
    ax.set_ylabel('State')
    save_chart(fig, 'state_transaction_analysis.png')
    plt.show()
    print('Observation: State-level transaction totals highlight geographic hotspots. \nInsight: Strong activity from specific states helps prioritize regional investor engagement and distribution support.')


In [ ]:
scheme_perf = datasets.get('scheme_performance', pd.DataFrame())
industry_folio = datasets.get('industry_folio_count', pd.DataFrame())

if not scheme_perf.empty:
    top30 = scheme_perf.nlargest(30, 'return_3yr_pct')
    bot30 = scheme_perf.nsmallest(30, 'return_3yr_pct')
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(top30['scheme_name'], top30['return_3yr_pct'], marker='o', label='Top 30', color='#2ca02c')
    ax.plot(bot30['scheme_name'], bot30['return_3yr_pct'], marker='o', label='Bottom 30', color='#d62728')
    ax.set_title('Top 30 vs Bottom 30 Schemes by 3-Year Return')
    ax.set_ylabel('3-Year Return (%)')
    ax.set_xticks([])
    ax.legend()
    save_chart(fig, 't30_vs_b30_analysis.png')
    plt.show()
    print('Observation: This comparison shows performance divergence across the scheme universe. \nInsight: Top and bottom performers can be used to screen risk and reward profiles.')

if not industry_folio.empty:
    industry_folio['month'] = pd.to_datetime(industry_folio['month'], errors='coerce')
    folio_growth = industry_folio.sort_values('month')
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(folio_growth['month'], folio_growth['total_folios_crore'], marker='o', label='Total Folios')
    ax.plot(folio_growth['month'], folio_growth['equity_folios_crore'], marker='o', label='Equity Folios')
    ax.plot(folio_growth['month'], folio_growth['debt_folios_crore'], marker='o', label='Debt Folios')
    ax.set_title('Folio Count Growth Over Time')
    ax.set_xlabel('Month')
    ax.set_ylabel('Folios (Crore)')
    ax.legend()
    save_chart(fig, 'folio_count_growth.png')
    plt.show()
    print('Observation: Folio growth reflects investor account expansion in equity and debt segments. \nInsight: Folio trends help estimate product adoption and market appetite.')


In [ ]:
if not scheme_perf.empty:
    top_perf = scheme_perf.sort_values('return_5yr_pct', ascending=False).head(12)
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.barplot(data=top_perf, x='return_5yr_pct', y='scheme_name', palette='coolwarm', ax=ax)
    ax.set_title('Top 12 Funds by 5-Year Return')
    ax.set_xlabel('5-Year Return (%)')
    ax.set_ylabel('Scheme Name')
    save_chart(fig, 'top_performing_funds.png')
    plt.show()
    print('Observation: The top performing funds deliver the strongest cumulative returns. \nInsight: These funds can be benchmarked for product positioning and portfolio selection.')

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.histplot(scheme_perf['sortino_ratio'].dropna(), bins=12, kde=False, color='#1f77b4', ax=ax)
    ax.set_title('Risk Distribution — Sortino Ratio')
    ax.set_xlabel('Sortino Ratio')
    ax.set_ylabel('Count')
    save_chart(fig, 'risk_distribution_sortino.png')
    plt.show()
    print('Observation: The distribution of Sortino ratios indicates risk-adjusted return profiles. \nInsight: Funds with higher Sortino ratios demonstrate better downside protection relative to returns.')


In [ ]:
if not scheme_perf.empty:
    expense_plot = scheme_perf.sort_values('expense_ratio_pct', ascending=False).head(12)
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.barplot(data=expense_plot, x='expense_ratio_pct', y='scheme_name', palette='rocket', ax=ax)
    ax.set_title('Highest Expense Ratios by Scheme')
    ax.set_xlabel('Expense Ratio (%)')
    ax.set_ylabel('Scheme Name')
    save_chart(fig, 'expense_ratio_analysis.png')
    plt.show()
    print('Observation: Expense ratios vary significantly across schemes. \nInsight: Higher expense ratios may reduce net investor returns and should be evaluated against fund performance.')

    fig, ax = plt.subplots(figsize=(12, 6))
    sns.scatterplot(data=scheme_perf, x='beta', y='alpha', hue='risk_grade', size='sharpe_ratio', sizes=(40, 180), palette='viridis', alpha=0.8, ax=ax)
    ax.set_title('Alpha vs Beta by Scheme')
    ax.set_xlabel('Beta')
    ax.set_ylabel('Alpha')
    save_chart(fig, 'alpha_beta_comparison.png')
    plt.show()
    print('Observation: Alpha/Beta comparison shows how funds balance risk and excess return. \nInsight: Positive alpha with moderate beta is typically preferred for balanced investors.')


In [ ]:
portfolio = datasets.get('portfolio_holdings', pd.DataFrame())

if not scheme_perf.empty:
    numeric_perf = scheme_perf.select_dtypes(include=[np.number]).dropna(axis=1, how='all')
    corr_matrix = numeric_perf.corr()
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', square=True, ax=ax, cbar_kws={'shrink': 0.8})
    ax.set_title('Correlation Matrix for Scheme Performance Metrics')
    save_chart(fig, 'correlation_matrix_performance.png')
    plt.show()
    print('Observation: Correlation matrix identifies relationships between performance and risk metrics. \nInsight: Strong correlations guide factor analysis for product comparisons.')

if not portfolio.empty:
    sector_allocation = portfolio.groupby('sector')['market_value_cr'].sum().sort_values(ascending=False).head(10)
    fig, ax = plt.subplots(figsize=(8, 8))
    wedges, texts, autotexts = ax.pie(sector_allocation, labels=sector_allocation.index, autopct='%1.1f%%', startangle=140, pctdistance=0.75)
    ax.set_title('Top Sector Allocation by Market Value')
    centre_circle = plt.Circle((0, 0), 0.45, color='white')
    fig.gca().add_artist(centre_circle)
    save_chart(fig, 'sector_allocation_donut.png')
    plt.show()
    print('Observation: Sector allocation shows concentration areas in portfolio holdings. \nInsight: Sector concentration risk is important for diversification assessment.')


In [ ]:
benchmark = datasets.get('benchmark_indices', pd.DataFrame())

if not benchmark.empty:
    benchmark['date'] = pd.to_datetime(benchmark['date'], errors='coerce')
    top_indices = benchmark['index_name'].value_counts().nlargest(5).index
    benchmark_plot = benchmark[benchmark['index_name'].isin(top_indices)]
    fig, ax = plt.subplots(figsize=(12, 6))
    for name, group in benchmark_plot.groupby('index_name'):
        group = group.sort_values('date')
        ax.plot(group['date'], group['close_value'], label=name)
    ax.set_title('Benchmark Index Comparison')
    ax.set_xlabel('Date')
    ax.set_ylabel('Close Value')
    ax.legend()
    save_chart(fig, 'benchmark_comparison.png')
    plt.show()
    print('Observation: Benchmark comparison reveals relative market levels across major indices. \nInsight: Use benchmark performance to contextualize scheme returns and allocation decisions.')

print('\n---\nKey conclusions:')
print('- NAV trend analysis reveals market movement across fund schemes.')
print('- AUM growth chart highlights the most dominant asset managers.')
print('- SIP inflow trend confirms systematic investing momentum.')
print('- Category heatmap identifies segment-specific inflow hotspots.')
print('- Demographic analysis supports targeted investor engagement.')
print('- Top vs bottom performer comparisons help identify differentiated fund strategies.')
print('- Sector and benchmark analysis provide a complete risk and return view.')
